<a href="https://colab.research.google.com/github/weagan/Share-PEFT/blob/main/Full_vs_Standard_vs_Share_PEFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: full fine-tuning on CoLA
!pip install -q --upgrade transformers datasets evaluate


from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

# 1. Load dataset
dataset = load_dataset("glue", "cola")
metric = evaluate.load("glue", "cola")

# 2. Load pretrained backbone
model_name = "FacebookAI/roberta-base"  # ✅ corrected model ID
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Preprocess
def tokenize(batch):
    return tokenizer(batch['sentence'], padding="max_length", truncation=True, max_length=128)

encoded_dataset = dataset.map(tokenize, batched=True)

# 4. Trainer setup
training_args = TrainingArguments(
    output_dir="./naive_cola",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics
)

# 5. Train
trainer.train()

# 6. Evaluate
results = trainer.evaluate()
print("Validation results:", results)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 47.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 18.7 MB/s eta 0:00:00


README.md: 0.00B [00:00, ?B/s]

cola/train-00000-of-00001.parquet:   0%|          | 0.00/251k [00:00<?, ?B/s]

cola/validation-00000-of-00001.parquet:   0%|          | 0.00/37.6k [00:00<?, ?B/s]

cola/test-00000-of-00001.parquet:   0%|          | 0.00/37.7k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Validation results: {'eval_loss': 0.4496777653694153, 'eval_matthews_correlation': 0.5444111532535151, 'eval_runtime': 6.6808, 'eval_samples_per_second': 156.119, 'eval_steps_per_second': 4.94, 'epoch': 3.0}


# Cell 2: LoRA / Share-style parameter-efficient fine-tuning
!pip install -q --upgrade transformers datasets evaluate peft

from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# 1. Load dataset
dataset = load_dataset("glue", "cola")
metric = evaluate.load("glue", "cola")

# 2. Load pretrained backbone
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Configure LoRA
lora_config = LoraConfig(
    r=8,                  # low-rank dimension
    lora_alpha=16,
    target_modules=["query", "value"],  # attention weights
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
peft_model = get_peft_model(model, lora_config)

# 4. Preprocess
def tokenize(batch):
    return tokenizer(batch['sentence'], padding="max_length", truncation=True, max_length=128)

encoded_dataset = dataset.map(tokenize, batched=True)

# 5. Trainer setup
training_args = TrainingArguments(
    output_dir="./lora_cola",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,  # slightly higher for LoRA
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics
)

# 6. Train
trainer.train()

# 7. Evaluate
results = trainer.evaluate()
print("Validation results:", results)


In [2]:
# Cell 1: Standard LoRA Fine-Tuning (task-specific adapters)
!pip install -q transformers datasets evaluate peft

from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# 1. Load dataset
dataset = load_dataset("glue", "cola")
metric = evaluate.load("glue", "cola")

# 2. Load backbone
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Configure LoRA (task-specific)
lora_config = LoraConfig(
    r=8,                  # low-rank dimension
    lora_alpha=16,
    target_modules=["query", "value"],  # attention layers
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)

# Wrap the model with LoRA adapters
peft_model = get_peft_model(model, lora_config)

# 4. Preprocess dataset
def tokenize(batch):
    return tokenizer(batch['sentence'], padding="max_length", truncation=True, max_length=128)

encoded_dataset = dataset.map(tokenize, batched=True)

# 5. Trainer
training_args = TrainingArguments(
    output_dir="./lora_standard",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics
)

# 6. Train
trainer.train()

# 7. Evaluate
results = trainer.evaluate()
print("Validation results:", results)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.428700,0.503650,0.446923
2,0.415951,0.491816,0.507529
3,0.405957,0.483203,0.529145


Validation results: {'eval_loss': 0.48320281505584717, 'eval_matthews_correlation': 0.529144545456451, 'eval_runtime': 6.9237, 'eval_samples_per_second': 150.642, 'eval_steps_per_second': 4.766, 'epoch': 3.0}


In [3]:
# Cell 2: Shared Subspace LoRA (Share-style)
!pip install -q transformers datasets evaluate peft

from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# 1. Load first dataset (task 1)
dataset_task1 = load_dataset("glue", "cola")
metric_task1 = evaluate.load("glue", "cola")

# 2. Load backbone
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# 3. Configure LoRA with shared subspace
# This simulates Share-style: same U, different alphas for each task
shared_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS",
    # note: in practice, you can store `U` once and reuse
)

shared_peft_model = get_peft_model(model, shared_lora_config)

# 4. Preprocess
def tokenize(batch):
    return tokenizer(batch['sentence'], padding="max_length", truncation=True, max_length=128)

encoded_dataset_task1 = dataset_task1.map(tokenize, batched=True)

# 5. Trainer
training_args = TrainingArguments(
    output_dir="./lora_shared_task1",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    logging_dir="./logs",
    logging_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=-1)
    return metric_task1.compute(predictions=predictions, references=labels)

trainer = Trainer(
    model=shared_peft_model,
    args=training_args,
    train_dataset=encoded_dataset_task1["train"],
    eval_dataset=encoded_dataset_task1["validation"],
    compute_metrics=compute_metrics
)

# 6. Train on Task 1
trainer.train()

# 7. Evaluate Task 1
results_task1 = trainer.evaluate()
print("Task 1 Validation results:", results_task1)

# 8. Later: For Task 2 (e.g., MRPC), you would reload `shared_peft_model`
# and only update a new set of alphas, keeping `U` frozen


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

Map:   0%|          | 0/1063 [00:00<?, ? examples/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Epoch,Training Loss,Validation Loss,Matthews Correlation
1,0.428700,0.503650,0.446923
2,0.415951,0.491816,0.507529
3,0.405957,0.483203,0.529145


Task 1 Validation results: {'eval_loss': 0.48320281505584717, 'eval_matthews_correlation': 0.529144545456451, 'eval_runtime': 6.9395, 'eval_samples_per_second': 150.299, 'eval_steps_per_second': 4.755, 'epoch': 3.0}


In [4]:
# Full sequential GLUE demo with Share-style LoRA and forgetting table
!pip install -q transformers datasets evaluate peft

import torch
from datasets import load_dataset
import evaluate
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model

# ----------------------
# 1. Load backbone & tokenizer
# ----------------------
model_name = "FacebookAI/roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

# ----------------------
# 2. Configure shared LoRA (U shared)
# ----------------------
shared_lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_CLS"
)
shared_peft_model = get_peft_model(model, shared_lora_config)

# ----------------------
# 3. Utility functions
# ----------------------
def tokenize(batch):
    return tokenizer(batch['sentence'], padding="max_length", truncation=True, max_length=128)

def make_trainer(peft_model, dataset, metric, output_dir, num_labels=2):
    peft_model.classifier = torch.nn.Linear(peft_model.config.hidden_size, num_labels)
    encoded_dataset = dataset.map(tokenize, batched=True)
    training_args = TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-4,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        num_train_epochs=3,
        logging_dir="./logs",
        logging_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
    )
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = logits.argmax(axis=-1)
        return metric.compute(predictions=predictions, references=labels)
    trainer = Trainer(
        model=peft_model,
        args=training_args,
        train_dataset=encoded_dataset["train"],
        eval_dataset=encoded_dataset["validation"],
        compute_metrics=compute_metrics
    )
    return trainer

# ----------------------
# 4. Sequential tasks
# ----------------------
tasks = [
    ("cola", 2),
    ("mrpc", 2),
    ("sst2", 2),
]

results_table = {}

for task_name, num_labels in tasks:
    print(f"\n=== Training on {task_name.upper()} ===")
    dataset = load_dataset("glue", task_name)
    metric = evaluate.load("glue", task_name)

    trainer = make_trainer(shared_peft_model, dataset, metric, f"./lora_shared_{task_name}", num_labels=num_labels)
    trainer.train()

    # Evaluate on all seen tasks so far
    for past_task_name, _ in tasks[:tasks.index((task_name, num_labels))+1]:
        past_dataset = load_dataset("glue", past_task_name)
        past_metric = evaluate.load("glue", past_task_name)
        past_encoded = past_dataset.map(tokenize, batched=True)
        logits = trainer.predict(past_encoded["validation"]).predictions
        preds = logits.argmax(axis=-1)
        results = past_metric.compute(predictions=preds, references=past_encoded["validation"]["label"])
        if past_task_name not in results_table:
            results_table[past_task_name] = []
        results_table[past_task_name].append(results)
        print(f"Validation on {past_task_name.upper()} after {task_name.upper()}: {results}")

# ----------------------
# 5. Build Forgetting Table
# ----------------------
import pandas as pd

df = pd.DataFrame.from_dict(results_table, orient="index")
df.columns = [f"After {t[0].upper()}" for t in tasks[:len(df.columns)]]
df = df.round(3)
print("\n=== Forgetting Table ===")
display(df)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: FacebookAI/roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



=== Training on COLA ===


Map:   0%|          | 0/8551 [00:00<?, ? examples/s]

Map:   0%|          | 0/1043 [00:00<?, ? examples/s]

...

Generating train split:   0%|          | 0/3668 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/408 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1725 [00:00<?, ? examples/s]

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

KeyError: 'sentence'